In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 04_kmeans_segmentation - Segmentación RFM
# MAGIC Clustering KMeans y etiquetado premium (2016-2018)

# COMMAND ----------

from pyspark.sql import functions as F
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

GOLD_PATH = "/Volumes/olist/olist_gold/gold/"
METRICS_PATH = "/Volumes/olist/olist_gold/metrics/"

# COMMAND ----------

# Crear volume metrics
try:
    spark.sql("CREATE VOLUME IF NOT EXISTS olist.olist_gold.metrics")
except:
    pass

# COMMAND ----------

# Cargar RFM y convertir a Pandas
rfm_pd = spark.read.format("delta").load(f"{GOLD_PATH}rfm_cutoff_20180930/").toPandas()

print(f"📥 {len(rfm_pd):,} clientes cargados\n")

# COMMAND ----------

# Verificar y limpiar NaNs
print("🔍 Verificando datos...\n")

print(f"NaNs por columna:")
print(rfm_pd[["recency", "frequency", "monetary"]].isnull().sum())
print()

# Eliminar filas con NaN en columnas RFM
rfm_clean = rfm_pd[["customer_id", "recency", "frequency", "monetary"]].dropna()

print(f"✅ Registros válidos: {len(rfm_clean):,} (eliminados: {len(rfm_pd) - len(rfm_clean):,})\n")

# COMMAND ----------

# Escalar RFM
X = rfm_clean[["recency", "frequency", "monetary"]].values
X_scaled = StandardScaler().fit_transform(X)

print("✅ Datos escalados\n")

# COMMAND ----------

# Calcular métricas para k=2..12
print("📊 Calculando métricas k=2..12...\n")

k_range = range(2, 13)
inertias = []
silhouettes = []
davies_bouldin = []
calinski_harabasz = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    
    # Inertia
    inertias.append(km.inertia_)
    
    # Silhouette (sample si > 10k)
    if len(X_scaled) > 10000:
        idx = pd.Series(range(len(X_scaled))).sample(10000, random_state=42).values
        sil = silhouette_score(X_scaled[idx], labels[idx])
    else:
        sil = silhouette_score(X_scaled, labels)
    silhouettes.append(sil)
    
    # Davies-Bouldin (menor es mejor)
    db = davies_bouldin_score(X_scaled, labels)
    davies_bouldin.append(db)
    
    # Calinski-Harabasz (mayor es mejor)
    ch = calinski_harabasz_score(X_scaled, labels)
    calinski_harabasz.append(ch)
    
    print(f"k={k:2d} | inertia={km.inertia_:>8,.0f} | sil={sil:.3f} | DB={db:.3f} | CH={ch:>8,.1f}")

print()

# COMMAND ----------

# Calcular tasa de cambio de inercia (método del codo)
print("📐 Análisis del codo:\n")

inertia_changes = []
for i in range(1, len(inertias)):
    change = abs(inertias[i] - inertias[i-1])
    rate = (change / inertias[i-1]) * 100
    inertia_changes.append(rate)
    print(f"k={i+2} → k={i+3}: cambio={rate:.2f}%")

print()

# COMMAND ----------

# Graficar todas las métricas
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Inertia
axes[0, 0].plot(k_range, inertias, 'bo-', linewidth=2)
axes[0, 0].set_xlabel('k', fontsize=10)
axes[0, 0].set_ylabel('Inertia', fontsize=10)
axes[0, 0].set_title('Método del Codo', fontsize=12, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

# Silhouette
axes[0, 1].plot(k_range, silhouettes, 'go-', linewidth=2)
axes[0, 1].set_xlabel('k', fontsize=10)
axes[0, 1].set_ylabel('Silhouette Score', fontsize=10)
axes[0, 1].set_title('Índice de Silueta (↑ mejor)', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].axhline(y=max(silhouettes), color='r', linestyle='--', alpha=0.5, label=f'Máx: {max(silhouettes):.3f}')
axes[0, 1].legend()

# Davies-Bouldin
axes[1, 0].plot(k_range, davies_bouldin, 'ro-', linewidth=2)
axes[1, 0].set_xlabel('k', fontsize=10)
axes[1, 0].set_ylabel('Davies-Bouldin Index', fontsize=10)
axes[1, 0].set_title('Davies-Bouldin (↓ mejor)', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].axhline(y=min(davies_bouldin), color='r', linestyle='--', alpha=0.5, label=f'Mín: {min(davies_bouldin):.3f}')
axes[1, 0].legend()

# Calinski-Harabasz
axes[1, 1].plot(k_range, calinski_harabasz, 'mo-', linewidth=2)
axes[1, 1].set_xlabel('k', fontsize=10)
axes[1, 1].set_ylabel('Calinski-Harabasz Score', fontsize=10)
axes[1, 1].set_title('Calinski-Harabasz (↑ mejor)', fontsize=12, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].axhline(y=max(calinski_harabasz), color='r', linestyle='--', alpha=0.5, label=f'Máx: {max(calinski_harabasz):,.0f}')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

# COMMAND ----------

# Seleccionar k óptimo (máximo silhouette)
k_opt = k_range[silhouettes.index(max(silhouettes))]

print(f"{'='*60}")
print("🎯 SELECCIÓN DE K ÓPTIMO")
print(f"{'='*60}")
print(f"Máximo Silhouette: k={k_opt} (score={max(silhouettes):.3f})")
print(f"Mínimo Davies-Bouldin: k={k_range[davies_bouldin.index(min(davies_bouldin))]} (score={min(davies_bouldin):.3f})")
print(f"Máximo Calinski-Harabasz: k={k_range[calinski_harabasz.index(max(calinski_harabasz))]} (score={max(calinski_harabasz):,.1f})")
print(f"\n✅ K seleccionado: {k_opt}\n")

# COMMAND ----------

# KMeans final
rfm_clean['cluster'] = KMeans(n_clusters=k_opt, random_state=42, n_init=10).fit_predict(X_scaled)

# Ordenar por monetary promedio
cluster_order = rfm_clean.groupby('cluster')['monetary'].mean().sort_values(ascending=False).index
cluster_map = {old: new+1 for new, old in enumerate(cluster_order)}

rfm_clean['cluster_ordered'] = rfm_clean['cluster'].map(cluster_map)
rfm_clean['is_premium'] = (rfm_clean['cluster_ordered'] == 1).astype(int)

premium_count = rfm_clean['is_premium'].sum()
print(f"✅ Clientes premium: {premium_count:,} ({premium_count/len(rfm_clean)*100:.1f}%)\n")

# COMMAND ----------

# Estadísticas por cluster
print("📊 Perfil de clusters:\n")

cluster_profile = rfm_clean.groupby('cluster_ordered').agg({
    'customer_id': 'count',
    'recency': 'mean',
    'frequency': 'mean',
    'monetary': 'mean'
}).round(2)
cluster_profile.columns = ['count', 'avg_recency', 'avg_frequency', 'avg_monetary']

print(cluster_profile)
print()

# COMMAND ----------

# Guardar segmentados (con mergeSchema para evitar error de schema mismatch)
spark.createDataFrame(rfm_clean) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .option("overwriteSchema", "true") \
    .save(f"{GOLD_PATH}customers_segmented_20180930/")

print(f"✅ Guardado: customers_segmented_20180930/\n")

# COMMAND ----------

# Guardar métricas completas
metrics = pd.DataFrame({
    'k': list(k_range),
    'inertia': inertias,
    'silhouette': silhouettes,
    'davies_bouldin': davies_bouldin,
    'calinski_harabasz': calinski_harabasz
})

spark.createDataFrame(metrics) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{METRICS_PATH}kmeans/")

print(f"✅ Guardado: metrics/kmeans/\n")

# COMMAND ----------

# Resumen final
print(f"{'='*60}")
print("✅ SEGMENTACIÓN COMPLETADA")
print(f"{'='*60}")
print(f"Total clientes: {len(rfm_clean):,}")
print(f"Clusters: {k_opt}")
print(f"Premium: {premium_count:,} ({premium_count/len(rfm_clean)*100:.1f}%)")
print(f"\nMétricas finales (k={k_opt}):")
print(f"  • Inertia: {inertias[k_opt-2]:,.0f}")
print(f"  • Silhouette: {silhouettes[k_opt-2]:.3f}")
print(f"  • Davies-Bouldin: {davies_bouldin[k_opt-2]:.3f}")
print(f"  • Calinski-Harabasz: {calinski_harabasz[k_opt-2]:,.1f}")
print(f"\nDistribución por cluster:")
print(rfm_clean['cluster_ordered'].value_counts().sort_index())